In [28]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [29]:
# 모듈 import
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.utils import plot_model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Concatenate, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [30]:
# 이미지 가져오기
train_dir = 'data/cat_dog_full/train'
validation_dir = 'data/cat_dog_full/validation'
test_dir = 'data/cat_dog_full/test'

In [31]:
# Parameter 설정
IMAGE_SIZE = 380
BATCH_SIZE = 32

In [32]:
# ImageDataGenerator 생성
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                  rotation_range=20,  
                                  width_shift_range=0.1,  
                                  height_shift_range=0.1, 
                                  zoom_range=0.2, 
                                  horizontal_flip=True,
                                  shear_range=0.1,
                                  fill_mode='nearest')    
validation_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [33]:
# ImageDataGenerator 설정
train_generator = train_datagen.flow_from_directory(
    train_dir,
    classes=['cats','dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=20,
    class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    classes=['cats','dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=20,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    classes=['cats','dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=20,
    class_mode='binary'
)

Found 14000 images belonging to 2 classes.
Found 6000 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


In [34]:
# model
model_base = EfficientNetB4(weights='imagenet',
                            include_top=False,
                            input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
for layer in model_base.layers:
    layer.trainable = False

In [35]:
model = Sequential()
model.add(model_base)
model.add(GlobalAveragePooling2D())
model.add(Dense(units=64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(rate=0.3))
model.add(Dense(units=1,
               activation='sigmoid'))

In [36]:
# model 설정
model.compile(optimizer=Adam(learning_rate=1e-4),
             loss='binary_crossentropy',
             metrics=['accuracy'])

In [37]:
es_callback = EarlyStopping(monitor='val_loss',
                           patience=5,
                           restore_best_weights=True,
                           verbose=1)
cp_callback = ModelCheckpoint(filepath='./efficientnetb4_weights.h5',
                             save_best_only=True,
                             save_weights_only=True,
                             monitor='val_accuracy',
                             verbose=1)

In [38]:
# 1차 학습 진행
model.fit(train_generator,
         steps_per_epoch=len(train_generator),
         epochs=30,
         validation_data=validation_generator,
         validation_steps=len(validation_generator),
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/30
700/700 [==============================] - ETA: 0s - loss: 0.0825 - accuracy: 0.9800     
Epoch 1: val_accuracy improved from -inf to 0.99483, saving model to ./efficientnetb4_weights.h5
700/700 [==============================] - 234s 323ms/step - loss: 0.0825 - accuracy: 0.9800 - val_loss: 0.0236 - val_accuracy: 0.9948
Epoch 2/30
700/700 [==============================] - ETA: 0s - loss: 0.0387 - accuracy: 0.9902  
Epoch 2: val_accuracy improved from 0.99483 to 0.99500, saving model to ./efficientnetb4_weights.h5
700/700 [==============================] - 226s 323ms/step - loss: 0.0387 - accuracy: 0.9902 - val_loss: 0.0176 - val_accuracy: 0.9950
Epoch 3/30
700/700 [==============================] - ETA: 0s - loss: 0.0290 - accuracy: 0.9934  
Epoch 3: val_accuracy improved from 0.99500 to 0.99517, saving model to ./efficientnetb4_weights.h5
700/700 [==============================] - 226s 322ms/step - loss: 0.0290 - accuracy: 0.9934 - val_loss: 0.0159 - val_accuracy: 0.9952
E

In [39]:
# Fine Tuning
model_base.trainable = True

for layer in model_base.layers[:-30]:
    layer.trainable = False

In [40]:
# model 재설정
model.compile(optimizer=Adam(learning_rate=1e-5),
             loss='binary_crossentropy',
             metrics=['accuracy'])

In [41]:
# model 재학습
model.fit(train_generator,
         steps_per_epoch=len(train_generator),
         epochs=30,
         validation_data=validation_generator,
         validation_steps=len(validation_generator),
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/30
700/700 [==============================] - ETA: 0s - loss: 0.0426 - accuracy: 0.9855     
Epoch 1: val_accuracy did not improve from 0.99633
700/700 [==============================] - 237s 323ms/step - loss: 0.0426 - accuracy: 0.9855 - val_loss: 0.0162 - val_accuracy: 0.9955
Epoch 2/30
700/700 [==============================] - ETA: 0s - loss: 0.0297 - accuracy: 0.9906  
Epoch 2: val_accuracy did not improve from 0.99633
700/700 [==============================] - 229s 328ms/step - loss: 0.0297 - accuracy: 0.9906 - val_loss: 0.0151 - val_accuracy: 0.9952
Epoch 3/30
700/700 [==============================] - ETA: 0s - loss: 0.0262 - accuracy: 0.9908  
Epoch 3: val_accuracy did not improve from 0.99633
700/700 [==============================] - 238s 340ms/step - loss: 0.0262 - accuracy: 0.9908 - val_loss: 0.0146 - val_accuracy: 0.9958
Epoch 4/30
700/700 [==============================] - ETA: 0s - loss: 0.0209 - accuracy: 0.9937  
Epoch 4: val_accuracy did not improve from 0.99

In [43]:
# 예측 수행
predict = model.evaluate(test_generator, verbose=1)
print(predict)  # [0.016437873244285583, 0.9954000115394592]

250/250 [==============================] - 26s 102ms/step - loss: 0.0164 - accuracy: 0.9954
[0.016437873244285583, 0.9954000115394592]
